## How Are Dropout Neurons Selected?

<img src="dropout-diag.png" height=300 width=500 />

Dropout randomly "turns off" neurons during training. Let's go into exactly how this selection works.

---

## The Mechanism

During each forward pass, for every neuron, a random number is drawn from a uniform distribution:

```
u ~ Uniform(0, 1)

if u < p:
    neuron = 0    ← dropped
else:
    neuron stays active
```

Where `p` is the **dropout probability** (e.g., p=0.5 means 50% chance of being dropped).

---

## It's a Bernoulli Draw

Each neuron independently samples from a **Bernoulli distribution** — essentially a biased coin flip:

```
mask ~ Bernoulli(1 - p)

output = activation * mask
```

The mask is a tensor of 0s and 1s, same shape as the layer output:

```
Activations:  [0.8,  1.2,  0.4,  0.9,  0.3]
Mask:         [  1,    0,    1,    0,    1 ]   ← random each forward pass
Output:       [0.8,  0.0,  0.4,  0.0,  0.3]
```

A completely new mask is sampled **every single forward pass** — so different neurons drop each time.

---

## Inverted Dropout (What PyTorch Actually Does)

There's a subtle problem — if you drop 50% of neurons during training, the activations at test time (where nothing is dropped) are roughly **2x larger** than what the network was trained on. This mismatch hurts performance.

The fix is **inverted dropout** — scale up the surviving neurons during training to compensate:

```
mask = Bernoulli(1 - p)
output = (activation * mask) / (1 - p)
                              ↑
                         scale up to maintain
                         expected value
```

Example with p=0.5:

```
Activations:  [0.8,  1.2,  0.4,  0.9]
Mask:         [  1,    0,    1,    0 ]
Scaled:       [1.6,  0.0,  0.8,  0.0]   ← surviving neurons doubled
```

At **test time**, dropout is simply turned off — no scaling needed because training already compensated.

---

## Visualizing Across Training Steps

Each forward pass gets a different random mask:

```
Step 1:   [■, □, ■, ■, □]   ← neurons 2,5 dropped
Step 2:   [□, ■, ■, □, ■]   ← neurons 1,4 dropped
Step 3:   [■, ■, □, ■, □]   ← neurons 3,5 dropped
Step 4:   [□, ■, □, ■, ■]   ← neurons 1,3 dropped

■ = active   □ = dropped
```

Over many steps, every neuron gets dropped sometimes and survives sometimes — so **all neurons are forced to learn independently.**

---

## Why Random Selection is the Point

The randomness is intentional — it prevents neurons from **co-adapting:**

```
Without dropout:
Neuron A learns to always correct for Neuron B's mistakes
→ they become dependent on each other
→ fragile, overfits

With dropout:
Neuron A can't rely on B (B might be dropped)
→ each neuron must be useful on its own
→ robust, generalizes better
```

This is equivalent to training an **ensemble of 2^N different networks** (where N = number of neurons) and averaging them at test time — which is why dropout works so well as a regularizer.

---

## Where Dropout is Applied

In CNNs, dropout is usually applied **after fully connected layers**, not conv layers. For conv layers, **spatial dropout** (dropping entire feature maps rather than individual neurons) is more effective.

Putting Batch Normalization (BN) before activation (Linear -&gt; BN -&gt; ReLU) is the original, generally recommended practice to ensure inputs to the non-linearity are Gaussian-distributed. Placing it after (Linear -&gt; ReLU -&gt; BN) is sometimes used for specific architectures or to stabilize non-Gaussian outputs, but it may alter normalized activations. 

• BN Before Activation (Standard): 

	• Goal: Normalize the raw, linear output ($W_x + b$) before the nonlinearity is applied. 
	• Advantage: Centers the data around the active region of the activation function (like sigmoid/tanh) or prevents "dying ReLU" issues by keeping data centered around zero before clipping. 
	• Result: More stable, faster training. 

• BN After Activation: 

	• Goal: Normalize the non-linear outputs (e.g., after ReLU). 
	• Advantage: Some empirical evidence suggests it may perform slightly better in specific deep residual networks (e.g., ResNet). 
	• Disadvantage: ReLU mapping all negative values to zero can create a skewed distribution (mean not zero).